# From uncertainty to action: which experiment should we run next?
**Optional AI/ML extension · teams of 2–3 · about 25 minutes**

Imagine that your team is training an ML model. You need to choose its **learning rate**, but every training run takes **two GPU-hours**. You can afford only eight runs.

A very small learning rate may learn too slowly. A very large one may make training unstable. The useful region lies somewhere in between, but we do not know where. Our goal is to find a low validation loss without paying to try every option.

We will use a **Gaussian process (GP)** to describe what we know about model performance at tried and untried learning rates. Then we will use that uncertainty to decide which run to buy next.

The ML problem, candidate learning rates, and compute budget will stay fixed. **Your task is to design the Bayesian optimizer:** decide what its GP should assume and how strongly its acquisition rule should explore. You will see how those choices change both the next experiment and the complete search path.

> This follows the [casino exercise](01_casino.ipynb). At each **Pause and predict**, discuss your answer before running the next cell.


In [ ]:
# Setup only: import libraries and configure paths
from pathlib import Path
import sys
import importlib

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'workshop').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import workshop.gp_optimization as gp_optimization
importlib.reload(gp_optimization)
from workshop.gp_optimization import (
    gp_posterior, lower_confidence_bound, rbf_kernel,
    run_bayesian_optimization,
    suggest_next_index,
)

DATA = ROOT / 'data' / 'public'
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## 1. The casino idea, enlarged

In the casino, one unknown number—win probability $p$—determined the result. Model tuning has an unknown **function**: each learning rate has its own expected validation loss.

| Casino | Model tuning |
|---|---|
| Unknown win probability $p$ | Unknown performance curve $f(x)$ |
| One play | One training run |
| Beta prior over $p$ | GP prior over possible curves |
| Posterior over $p$ | Posterior over curves |
| Decide whether to keep playing | Decide which learning rate to train next |

For a fast and reproducible workshop, a CSV acts as our training service. It contains fixed, **simulated but realistic** results for 33 possible learning rates. We reveal a row only when we pay for that run. No real GPU job is launched, and all compute costs in this exercise are illustrative.


In [ ]:
all_runs = pd.read_csv(DATA / 'gp_tuning_results.csv')
candidate_x = all_runs.log10_learning_rate.to_numpy()
cached_loss = all_runs.validation_loss.to_numpy()

BUDGET = 8
HOURS_PER_RUN = 2
initial_log_rates = [-4.75, -3.50, -1.25]
initial_indices = [
    int(np.flatnonzero(np.isclose(candidate_x, value))[0])
    for value in initial_log_rates
]
observed_indices = initial_indices.copy()

initial_results = all_runs.iloc[observed_indices][
    ['learning_rate', 'log10_learning_rate', 'validation_loss']
].copy()
initial_results.columns = ['Learning rate', 'log10 learning rate', 'Validation loss']
initial_results['Learning rate'] = initial_results['Learning rate'].map(lambda value: f'{value:.2e}')
display(initial_results.reset_index(drop=True))

plt.scatter(candidate_x[observed_indices], cached_loss[observed_indices],
            s=80, color='#2878B5', zorder=3)
plt.xlabel('log10(learning rate)')
plt.ylabel('Validation loss (lower is better)')
plt.title('The three training runs we have paid for', loc='left')
plt.xlim(candidate_x.min(), candidate_x.max())
plt.ylim(.22, .55)
plt.grid(alpha=.15)
plt.show()


**Pause and predict:** where do you think the best learning rate lies? Where are you most uncertain? If you could buy one run now, which value would you try?


## 2. Design your Gaussian process

A Gaussian process is a probability distribution over functions. Before seeing results, it describes many performance curves that we consider plausible. After observing runs, Bayes' rule gives more weight to curves that agree with those results.

With only three observations, we should not expect the GP to determine its assumptions reliably for us. The code cell below is your control panel. Each setting answers a question about the optimization problem, not about the model architecture.

| Control | Units here | What are you telling the optimizer? | Values worth trying |
|---|---|---|---|
| `PRIOR_MEAN` | validation loss | What loss is plausible before nearby evidence exists? | `0.35`, `0.45`, `0.55` |
| `SIGNAL_STD` | validation loss | How much can the average-loss curve vary vertically? | `0.05`, `0.10`, `0.16` |
| `LENGTH_SCALE` | $\log_{10}$(learning rate) | How far does information travel? Smaller allows sharper bends. | `0.25`, `0.65`, `1.50` |
| `NOISE_STD` | validation loss | How much might repeated training runs disagree? | `0.005`, `0.015`, `0.040` |
| `EXPLORATION` | none (dimensionless) | How much should uncertainty attract the next experiment? | `0`, `1.25`, `2.50` |

The first four controls define the GP; `EXPLORATION` belongs to the acquisition rule that will use it. We keep the kernel family fixed as a smooth radial basis function (RBF), while `LENGTH_SCALE` controls how it behaves. A length scale of `0.65` corresponds to learning rates differing by a factor of $10^{0.65} \approx 4.5$; at that separation, their RBF prior correlation is $e^{-1/2} \approx 0.61$. `SIGNAL_STD**2` is the kernel variance, so it has squared-loss units. If you standardize inputs or outputs in your own project, the settings use those standardized units instead.

There is no universally correct setting. In a real project you would use domain knowledge, repeated runs, historical experiments, and sensitivity checks—not tune these values against hidden answers. If your sampled prior curves include impossible loss values, treat that as useful feedback that the prior is too broad or the objective needs a transformation.

**Your experiment:** change **one control at a time**, predict what will happen, and rerun from this cell through the next-suggestion plot. Then choose one design your team can explain before completing the eight-run search.


In [ ]:
# YOUR OPTIMIZER DESIGN: edit these values, then rerun the cells below.
PRIOR_MEAN = 0.45      # validation-loss units; baseline away from observations
SIGNAL_STD = 0.10      # validation-loss units; vertical curve variation
LENGTH_SCALE = 0.65    # log10-learning-rate units; smaller = sharper bends
NOISE_STD = 0.015      # validation-loss units; run-to-run variation
EXPLORATION = 1.25     # dimensionless; 0 = predicted loss only

gp_settings = dict(
    prior_mean=PRIOR_MEAN,
    signal_std=SIGNAL_STD,
    length_scale=LENGTH_SCALE,
    noise_std=NOISE_STD,
)
x_plot = np.linspace(candidate_x.min(), candidate_x.max(), 500)
Z90 = 1.645

def posterior_at(indices, query, settings=None):
    settings = gp_settings if settings is None else settings
    return gp_posterior(
        candidate_x[indices], cached_loss[indices], query, **settings
    )

def plot_gp(ax, indices, title, next_index=None, settings=None):
    active_settings = gp_settings if settings is None else settings
    mean, latent_std, _ = posterior_at(indices, x_plot, active_settings)
    ax.fill_between(
        x_plot, mean - Z90 * latent_std, mean + Z90 * latent_std,
        color='#2878B5', alpha=.25, label='90% band for average loss',
    )
    ax.plot(x_plot, mean, color='#205E86', linewidth=2.5,
            label='GP posterior mean')
    ax.scatter(candidate_x[indices], cached_loss[indices], color='#172B4D',
               edgecolor='white', linewidth=.7, s=58, zorder=4,
               label='Observed runs')
    if next_index is not None:
        next_mean = posterior_at(indices, candidate_x, active_settings)[0][next_index]
        ax.scatter(candidate_x[next_index], next_mean, marker='*', s=180,
                   color='#D97706', zorder=5, label='Suggested next run')
    ax.set(xlim=(candidate_x.min(), candidate_x.max()),
           xlabel='log10(learning rate)', ylabel='Validation loss', title=title)
    ax.margins(y=.08)
    ax.grid(alpha=.15)

# Draw possible curves from your prior so its assumptions are visible.
prior_x = np.linspace(candidate_x.min(), candidate_x.max(), 180)
prior_covariance = rbf_kernel(
    prior_x, prior_x, LENGTH_SCALE, SIGNAL_STD
)
prior_cholesky = np.linalg.cholesky(
    prior_covariance + np.eye(len(prior_x)) * 1e-10
)
prior_samples = (PRIOR_MEAN +
                 np.random.default_rng(7).normal(size=(5, len(prior_x)))
                 @ prior_cholesky.T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True,
                         layout='constrained')
for sample_number, sample in enumerate(prior_samples):
    axes[0].plot(prior_x, sample, color='#7C3AED', alpha=.62, linewidth=1.7,
                 label='Random draw from GP prior' if sample_number == 0 else None)
axes[0].axhline(PRIOR_MEAN, color='#475569', linestyle='--', linewidth=2,
                label='Prior mean')
axes[0].set(xlim=(candidate_x.min(), candidate_x.max()),
            xlabel='log10(learning rate)', ylabel='Validation loss',
            title='Before data: curves your settings allow')
axes[0].grid(alpha=.15)
plot_gp(axes[1], observed_indices, 'After three runs: updated belief')
legend_items = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=3, frameon=False)
plt.show()


The left panel makes your prior concrete. `PRIOR_MEAN` moves its center, `SIGNAL_STD` changes its vertical spread, and `LENGTH_SCALE` changes how quickly its curves can bend. `NOISE_STD` matters after observations arrive: larger values tell the GP to trust each individual run less.

In the right panel, the blue band describes **epistemic uncertainty** about the average loss function because we have not evaluated many learning rates. It is not a range for one future noisy run. `NOISE_STD` still matters because it prevents the GP from treating each observed training result as exact.

**Pause and predict:** choose one control to change. Sketch or describe how you expect the two panels to change, then rerun the cell. Did the result match your explanation? Note that `EXPLORATION` will not change either GP panel—its effect appears in the next section.


## 3. Let your optimizer choose the next experiment

We need a repeatable rule for buying the next run. A common acquisition rule constructs an **optimistic estimate of the average validation loss**:

$$\text{optimistic average loss}(x) = \text{predicted average loss}(x) - \kappa \times \text{uncertainty about that average}(x).$$

This asks: **how low could the average loss plausibly be?** A low prediction makes a learning rate attractive. High uncertainty can also make it worth trying, because the true average might be better than our current prediction. Your `EXPLORATION` value is $\kappa$: `0` ignores uncertainty, while larger values give uncertain candidates a bigger bonus.

The mean and uncertainty here describe the latent **average validation loss**, not the noisy result of one future training run. The rule is conventionally called a *lower confidence bound (LCB)*, but here it is a ranking heuristic rather than a guaranteed bound.


In [ ]:
candidate_mean, candidate_latent_std, _ = posterior_at(
    observed_indices, candidate_x
)
acquisition = lower_confidence_bound(
    candidate_mean, candidate_latent_std, EXPLORATION
)
next_index = suggest_next_index(
    candidate_mean, candidate_latent_std, observed_indices, EXPLORATION
)
visible_acquisition = acquisition.copy()
visible_acquisition[observed_indices] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), layout='constrained')
plot_gp(axes[0], observed_indices, 'What the GP currently believes', next_index)
axes[1].plot(candidate_x, visible_acquisition, marker='.', color='#7C3AED',
             label='Predicted average loss − uncertainty bonus')
axes[1].scatter(candidate_x[next_index], acquisition[next_index], marker='*',
                s=180, color='#D97706', zorder=4,
                label='Suggested next run')
axes[1].set(xlabel='log10(learning rate)',
            ylabel='Optimistic average-loss score (lower is tried first)',
            title='Where should we train next?')
axes[1].grid(alpha=.15)
legend_items = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=3, frameon=False)
plt.show()

suggested_rate = all_runs.iloc[next_index].learning_rate
print(f'The GP suggests log10(learning rate) = {candidate_x[next_index]:.3f}')
print(f'That is a learning rate of {suggested_rate:.2e}.')


Compare the suggestion with your intuition. If you change `EXPLORATION` in the control panel and rerun, the GP belief stays the same but this ranking—and possibly the suggestion—changes. If you change a GP setting, both panels can change.

**Pause and experiment:** try at least two optimizer designs while the result is still hidden. Record the first suggestion from each. Then settle on one design your team can justify, rerun from the control panel, and reveal its next result below.


In [ ]:
observed_indices.append(next_index)
new_run = all_runs.iloc[next_index]
print(f'New run: learning rate {new_run.learning_rate:.2e}')
print(f'Observed validation loss: {new_run.validation_loss:.4f}')
print(f'Best loss before this run: {cached_loss[initial_indices].min():.4f}')

fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
plot_gp(ax, observed_indices, 'Posterior after buying one more run')
ax.scatter(candidate_x[next_index], cached_loss[next_index], marker='*',
           color='#D97706', s=190, zorder=5, label='New result')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.16), ncol=3, frameon=False)
plt.show()


## 4. Run the optimizer you designed

Now commit to one set of GP and acquisition settings. We will repeat the same cycle: update the GP, score every untried candidate, buy the most attractive run, and update again. The cell below replays **your policy** from the same three starting runs until the eight-run budget is exhausted.

The ML setup, candidate learning rates, starting evidence, and budget remain fixed; only the optimizer design is yours. The policy uses only results already revealed at each step and cannot peek at the other cached losses. If several teams are working together, choose different settings and compare the paths afterward.

**Before running:** write one sentence explaining your most consequential setting. What behavior do you expect it to cause over all five remaining decisions?


In [ ]:
selected_indices = run_bayesian_optimization(
    candidate_x, cached_loss, initial_indices, BUDGET,
    **gp_settings, exploration=EXPLORATION,
)
optimizer_design = pd.DataFrame({
    'Control': ['Prior mean', 'Signal std', 'Length scale',
                'Noise std', 'Exploration κ'],
    'Your value': [PRIOR_MEAN, SIGNAL_STD, LENGTH_SCALE,
                   NOISE_STD, EXPLORATION],
})
display(optimizer_design.style.format({'Your value': '{:.3f}'}).hide(axis='index'))

history = all_runs.iloc[selected_indices][
    ['learning_rate', 'log10_learning_rate', 'validation_loss']
].copy().reset_index(drop=True)
history.insert(0, 'Decision', np.arange(1, len(history) + 1))
history.insert(1, 'Chosen by', ['Initial design'] * len(initial_indices) +
               ['GP acquisition'] * (BUDGET - len(initial_indices)))
history['Best loss so far'] = history.validation_loss.cummin()
shown_history = history.copy()
shown_history['learning_rate'] = shown_history.learning_rate.map(lambda value: f'{value:.2e}')
display(shown_history.round({'log10_learning_rate': 3,
                             'validation_loss': 4, 'Best loss so far': 4}))

snapshot_sizes = [3, 4, 6, 8]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True,
                         layout='constrained')
for ax, count in zip(axes.flat, snapshot_sizes):
    next_choice = selected_indices[count] if count < BUDGET else None
    plot_gp(ax, selected_indices[:count], f'After {count} runs', next_choice)
    if count > len(initial_indices):
        newest = selected_indices[count - 1]
        ax.scatter(candidate_x[newest], cached_loss[newest], marker='*',
                   s=190, color='#FACC15', edgecolor='#854D0E',
                   linewidth=1, zorder=6, label='Newest observed result')
legend_items = {}
for ax in axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=4, frameon=False)
fig.suptitle('Your settings shape the complete sequence of experiments', fontsize=15)
plt.show()


## 5. Final reveal: what did your optimizer do?

We can now reveal all 33 cached results for a retrospective check. The left panel shows where your optimizer spent its eight-run budget; the yellow star is its final evaluation. The gray curve was hidden during every decision. The right panel compares the best result your optimizer observed with 5,000 random continuations that receive the same three initial runs and the same five-run remaining budget.

This is a fairer comparison than showing one lucky or unlucky random search. It is still one simulated tuning problem, not a score for choosing GP settings and not proof that Bayesian optimization always wins.


In [ ]:
best_search_index = min(selected_indices, key=lambda index: cached_loss[index])
best_search_loss = cached_loss[best_search_index]

rng = np.random.default_rng(42)
available = np.setdiff1d(np.arange(len(candidate_x)), initial_indices)
random_best_losses = []
for _ in range(5000):
    extra = rng.choice(available, BUDGET - len(initial_indices), replace=False)
    random_indices = np.r_[initial_indices, extra]
    random_best_losses.append(cached_loss[random_indices].min())
random_best_losses = np.asarray(random_best_losses)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), layout='constrained')
axes[0].plot(candidate_x, cached_loss, color='#94A3B8', marker='.',
             linewidth=1.5, label='Full table (revealed afterward)')
points = axes[0].scatter(
    candidate_x[selected_indices], cached_loss[selected_indices],
    c=np.arange(1, BUDGET + 1), cmap='viridis', s=75, edgecolor='white',
    linewidth=.7, zorder=4, label='Your GP-guided search',
)
final_index = selected_indices[-1]
axes[0].scatter(candidate_x[final_index], cached_loss[final_index],
                marker='*', s=230, color='#FACC15', edgecolor='#854D0E',
                linewidth=1, zorder=5, label='Final (8th) evaluation')
axes[0].set(xlabel='log10(learning rate)', ylabel='Validation loss',
            title='Where your optimizer spent its budget')
axes[0].grid(alpha=.15)
axes[0].legend(frameon=False, fontsize=9)
colorbar = fig.colorbar(points, ax=axes[0], pad=.02)
colorbar.set_label('Evaluation order')

axes[1].hist(random_best_losses, bins=20, color='#94A3B8', edgecolor='white')
axes[1].axvline(best_search_loss, color='#2878B5', linewidth=2.5,
                label=f'Your result ({best_search_loss:.4f})')
axes[1].set(xlabel='Best validation loss after 8 runs',
            ylabel='Number of random searches',
            title='Same budget, random choices')
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=.15)
plt.show()

random_match_rate = np.mean(random_best_losses <= best_search_loss)
print(f'Best result observed by your optimizer: learning rate '
      f'{all_runs.iloc[best_search_index].learning_rate:.2e}, '
      f'validation loss {best_search_loss:.4f}')
print(f'Random search matched or beat your result in {random_match_rate:.1%} of replays.')
print(f'Illustrative compute used: {BUDGET * HOURS_PER_RUN} GPU-hours')
print(f'Illustrative full-grid compute: {len(all_runs) * HOURS_PER_RUN} GPU-hours')


The optimizer can find a strong configuration without reconstructing every detail of the performance curve. Its uncertainty is useful because it changes **where compute is spent**. Compare your path with another team's: different defensible settings can collect different evidence.

But the calculation does not establish that our assumptions were appropriate:

- A smooth GP can miss a narrow, isolated optimum.
- The noise level may differ across learning rates.
- An acquisition policy can make an unlucky sequence of choices.
- Reusing validation results for selection makes the validation score optimistic. After choosing a configuration, retrain it and evaluate it once on untouched test data.

Just as sampler diagnostics cannot validate the casino model, one successful optimization replay cannot validate its kernel or data-generating assumptions.


## 6. Explain the optimizer you designed

The decision you made was the **design of a sequential optimization method**, not a judgment about a particular ML model. Finish these sentences with your team:

- Our prior mean and signal scale represented …
- On the $\log_{10}$ scale, our length scale said that learning rates about … apart should …
- Our noise setting treated each training result as …
- We chose an exploration value of … because …
- When we changed …, the next suggestion or search path changed because …
- Before trusting this design on our own project, we would check …

A useful design is not the one that happens to win this hidden table. It is one whose assumptions were reasonable **before the reveal**, whose behavior you understand, and whose sensitivity you have checked.

### Take the loop to your own project

This notebook replays cached results only to avoid real GPU cost. In a project, the sequential loop is the same, but the newly proposed point goes to your actual experiment:

```python
candidates = np.asarray([...], dtype=float)
observed_indices = [...]       # a small, space-filling initial design
observed_scores = [...]        # results in the same order

while len(observed_indices) < budget:
    mean, uncertainty, _ = gp_posterior(
        candidates[observed_indices],
        observed_scores,
        candidates,
        **gp_settings,
    )
    next_index = suggest_next_index(
        mean, uncertainty, observed_indices, EXPLORATION
    )
    next_score = run_your_expensive_experiment(candidates[next_index])
    observed_indices.append(next_index)
    observed_scores.append(next_score)
```

Before using the loop, define the candidates, objective, budget, plausible variation, smoothness, and repeatability. Express the objective so lower is better (for example, negate a reward you want to maximize). The small helper here is deliberately transparent and handles one numeric input on a finite grid. For several parameters, categories, constraints, or parallel experiments, use a production Bayesian-optimization implementation—but keep the same habit of stating and stress-testing its assumptions.

The practical lesson is not that every ML problem needs a Gaussian process. It is that representing uncertainty can help an AI/ML system decide **what evidence to collect next**, especially when labels, experiments, API calls, or training runs are expensive.


## Optional: what if our smoothness assumption changes?

The length scale controls how far one observation influences predictions. A short length scale permits rapidly changing curves; a long one assumes a very smooth curve. This controlled comparison temporarily tries three values, then lets each GP complete the full eight-run optimization.

Your other control-panel values stay fixed, as do the starting runs, acquisition rule, candidate results, and budget. Only the length scale changes.

**Pause and predict:** which assumption will leave the most uncertainty between observed learning rates? Will all three searches finish with the same chosen learning rate?


In [ ]:
length_scales = [0.25, 0.65, 1.50]
assumption_names = {0.25: 'Short', 0.65: 'Medium', 1.50: 'Long'}
searches_by_length_scale = {}
comparison_rows = []

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True,
                         layout='constrained')
for ax, length_scale in zip(axes, length_scales):
    alternative = {**gp_settings, 'length_scale': length_scale}
    selected = run_bayesian_optimization(
        candidate_x, cached_loss, initial_indices, BUDGET,
        **alternative, exploration=EXPLORATION,
    )
    searches_by_length_scale[length_scale] = selected
    suggestion = selected[len(initial_indices)]
    best_index = min(selected, key=lambda index: cached_loss[index])
    comparison_rows.append({
        'Length scale': length_scale,
        'Assumption': assumption_names[length_scale],
        'First GP choice': f'{all_runs.iloc[suggestion].learning_rate:.2e}',
        'Final (8th) learning rate': f'{all_runs.iloc[selected[-1]].learning_rate:.2e}',
        'Best observed learning rate': f'{all_runs.iloc[best_index].learning_rate:.2e}',
        'Best observed loss': cached_loss[best_index],
    })
    plot_gp(ax, initial_indices,
            f'{assumption_names[length_scale]} length scale = {length_scale:.2f}'
            f'\nfirst GP choice: x = {candidate_x[suggestion]:.3f}',
            suggestion, alternative)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3, frameon=False)
fig.suptitle('Same evidence, different beliefs about smoothness', fontsize=15)
plt.show()

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({'Length scale': 2, 'Best observed loss': 4}))


### Follow every step of each optimization

Each sequence begins with the same three observations. The next five panels show the posterior after each GP-guided result arrives. A **yellow star marks the newest result** in that panel; in the last panel it therefore marks the eighth and final evaluation.

Only results available at that step are plotted. The unrevealed cached candidates do not appear in these sequences.


In [ ]:
snapshot_counts = range(len(initial_indices), BUDGET + 1)
for length_scale in length_scales:
    alternative = {**gp_settings, 'length_scale': length_scale}
    selected = searches_by_length_scale[length_scale]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), sharex=True, sharey=True,
                             layout='constrained')
    for ax, count in zip(axes.flat, snapshot_counts):
        title = 'Starting evidence: 3 runs' if count == len(initial_indices) else f'After {count} runs'
        plot_gp(ax, selected[:count], title, settings=alternative)
        if count > len(initial_indices):
            newest = selected[count - 1]
            ax.scatter(candidate_x[newest], cached_loss[newest], marker='*',
                       s=190, color='#FACC15', edgecolor='#854D0E',
                       linewidth=1, zorder=6, label='Newest GP-chosen result')
    legend_items = {}
    for ax in axes.flat:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            legend_items.setdefault(label, handle)
    fig.legend(legend_items.values(), legend_items.keys(),
               loc='outside lower center', ncol=4, frameon=False)
    fig.suptitle(
        f'{assumption_names[length_scale]} smoothness assumption '
        f'(length scale = {length_scale:.2f})', fontsize=15,
    )
    plt.show()


### Retrospective comparison

Because Section 5 has already revealed the complete cached table, we can use it as gray background for a retrospective comparison. The optimization did **not** see those gray points while choosing. The yellow star marks the final evaluation—not a claimed global optimum.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharex=True, sharey=True,
                         layout='constrained')
for ax, length_scale in zip(axes, length_scales):
    selected = searches_by_length_scale[length_scale]
    best_index = min(selected, key=lambda index: cached_loss[index])
    ax.plot(candidate_x, cached_loss, color='#A8B3C3', marker='.',
            linewidth=1.4, label='Full table (revealed afterward)')
    ax.scatter(candidate_x[initial_indices], cached_loss[initial_indices],
               s=64, facecolor='white', edgecolor='#172B4D', linewidth=1.5,
               zorder=4, label='Three starting runs')
    ax.scatter(candidate_x[selected[3:]], cached_loss[selected[3:]],
               s=64, color='#2878B5', edgecolor='white', linewidth=.7,
               zorder=4, label='Five GP-chosen runs')
    final_index = selected[-1]
    ax.scatter(candidate_x[final_index], cached_loss[final_index],
               marker='*', s=220, color='#FACC15', edgecolor='#854D0E',
               linewidth=1, zorder=6, label='Final (8th) evaluation')
    ax.set(xlim=(candidate_x.min(), candidate_x.max()), ylim=(.245, .515),
           xlabel='log10(learning rate)', ylabel='Validation loss',
           title=f'{assumption_names[length_scale]} length scale = {length_scale:.2f}'
                 f'\nbest observed loss: {cached_loss[best_index]:.4f}')
    ax.grid(alpha=.15)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=4, frameon=False)
fig.suptitle('The smoothness assumption changes the entire search path', fontsize=15)
plt.show()


On this fixed replay, the three assumptions collect different evidence and finish with different best observed losses. The short length scale treats nearby candidates as only weakly related, while the long length scale shares information across a much wider range.

This is a sensitivity analysis, not a reason to select whichever length scale looks best after seeing this table. It demonstrates why checking only the first suggested point is insufficient: an assumption changes the evidence collected later, and those observations change every subsequent decision. In practice, kernel choices should be justified with domain knowledge and checked on additional tasks or historical experiments.
